# Formula 1 Exploratory Data Analysis (EDA)

In this notebook, we perform exploratory data analysis and visualize key insights from the cleaned Formula 1 historical dataset.

## Analytical Questions:
1. Which constructors have been the most dominant in F1 history (by total wins)?
2. How do driver stats compare (wins, podiums, poles)?
3. How have pit stop durations evolved over time, and which teams are the fastest?
4. How do teammate battles look (head-to-head qualifying and finishing positions)?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = '../Data'

## 1. Load Cleaned Data
Let's write helper functions to quickly load the cleaned datasets (implementing the same logic as our data loader).

In [ ]:
def load_csv(name):
    return pd.read_csv(os.path.join(DATA_DIR, name)).replace(r'\\N', np.nan, regex=True).replace(r'\\N', np.nan)

drivers = load_csv('drivers.csv')
drivers['driver_name'] = drivers['forename'] + ' ' + drivers['surname']

races = load_csv('races.csv')
races['year'] = pd.to_numeric(races['year'])

constructors = load_csv('constructors.csv').rename(columns={'name': 'constructor_name'})

results = load_csv('results.csv')
results['positionOrder'] = pd.to_numeric(results['positionOrder'])
results['grid'] = pd.to_numeric(results['grid'])
results['points'] = pd.to_numeric(results['points'])

# Merge master dataset
merged = results.merge(races[['raceId', 'year', 'name']], on='raceId', how='left')
merged = merged.merge(drivers[['driverId', 'driver_name']], on='driverId', how='left')
merged = merged.merge(constructors[['constructorId', 'constructor_name']], on='constructorId', how='left')
merged.head()

## 2. Most Dominant Constructors
Let's find the teams with the most race victories in F1 history (where finish position is 1).

In [ ]:
wins_by_constructor = merged[merged['positionOrder'] == 1]['constructor_name'].value_counts().reset_index()
wins_by_constructor.columns = ['constructor_name', 'wins']

sns.barplot(data=wins_by_constructor.head(10), x='wins', y='constructor_name', hue='constructor_name', legend=False, palette='viridis')
plt.title('Top 10 Constructors by Total Victories')
plt.xlabel('Number of Wins')
plt.ylabel('Constructor')
plt.tight_layout()
plt.show()

Ferrari leads by a significant margin, followed by McLaren, Mercedes, Williams, and Red Bull.

## 3. Pit Stop Efficiency Over Time
Let's look at the pit stop dataset to analyze average stop times per year. We want to see how the introduction of refueling bans and technology has speeded up stops.

In [ ]:
pit_stops = load_csv('pit_stops.csv')
pit_stops['milliseconds'] = pd.to_numeric(pit_stops['milliseconds'], errors='coerce')
pit_stops['duration'] = pd.to_numeric(pit_stops['duration'], errors='coerce')

# Merge with race year
pit_stops_merged = pit_stops.merge(races[['raceId', 'year']], on='raceId', how='left')

# Filter out outlier pit stops (stops > 50s are usually repairs rather than regular stops)
clean_stops = pit_stops_merged[pit_stops_merged['duration'] < 50]

# Average pit stop time by year
stops_by_year = clean_stops.groupby('year')['duration'].mean().reset_index()

sns.lineplot(data=stops_by_year, x='year', y='duration', marker='o', color='crimson', linewidth=2.5)
plt.title('Average Pit Stop Duration by Year (Stops < 50 seconds)')
plt.xlabel('Year')
plt.ylabel('Average Duration (seconds)')
plt.xticks(stops_by_year['year'].unique())
plt.show()

## 4. Teammate Battles
Comparing teammates is the ultimate metric in F1 since they drive the identical car. Let's write a script to compute the head-to-head finish comparison of teammate pairs for a selected season and team. We can prototype this for Mercedes in 2021 (Hamilton vs. Bottas).

In [ ]:
# Filter for 2021 Mercedes results
m2021 = merged[(merged['year'] == 2021) & (merged['constructor_name'] == 'Mercedes')].copy()

# Let's see the distribution of finishes for the two drivers
sns.boxplot(data=m2021, x='driver_name', y='positionOrder', hue='driver_name', legend=False, palette='Set2')
plt.title('Mercedes 2021 Driver Finishes Distribution (Lower is Better)')
plt.xlabel('Driver')
plt.ylabel('Finish Position')
plt.show()

Hamilton has a lower (better) median and lower range, showing his dominance in the teammate battle during the 2021 season.

## Conclusion
This exploratory analysis demonstrates key F1 trends: team historical dominance, pit stop efficiency, and driver/teammate battles. These form the core visualizations of the Streamlit dashboard.